# Task 2 - Manual Error Analysis

This notebook performs the required qualitative review of **20 frozen official-test examples** for the BiGRU-Attention sentiment classifier.

The review set contains exactly:

- 5 confident false positives
- 5 confident false negatives
- 5 near-threshold errors
- 5 slice-specific errors

The row selection was frozen and pushed to Git **before review text was accessed**. The raw review-text evidence was subsequently frozen and pushed in a separate commit before qualitative interpretation was authored.

This notebook is strictly post-hoc and descriptive. No observation in this notebook is used to modify model weights, architecture, hyperparameters, decision threshold, preprocessing, robustness-slice definitions, or checkpoint selection.


## Interpretation boundary

The purpose of this notebook is to understand **why representative errors may have occurred**, not to claim causal attribution from model internals.

Therefore:

- phrases such as *consistent with*, *plausible failure mode*, and *may reflect* are used deliberately;
- no attention weights, saliency maps, or token-level causal explanations are available here;
- apparent label-text disagreements are documented as **possible annotation or source-context issues**, not automatically declared mislabeled;
- all conclusions are based on the already-frozen test predictions and already-frozen raw review snapshot;
- these observations must not feed back into model, threshold, hyperparameter, or slice tuning.


In [1]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display


ROOT = Path.cwd()

if ROOT.name != "Lab1":
    raise RuntimeError(
        "Run this notebook from the DATA 266 Lab1 repository root."
    )

MEMBER = ROOT / "task2_sentiment" / "Anshika_Goel"

SNAPSHOT_PATH = (
    MEMBER
    / "outputs"
    / "manual_error_analysis"
    / "manual_review_text_snapshot.json"
)

EXPECTED_SNAPSHOT_CANONICAL_SHA = (
    "916ec84712feabdfc0e1feb26c0ba0ed"
    "d09ad51209aee444e3d7341f6551bff8"
)

EXPECTED_SELECTION_CANONICAL_SHA = (
    "489fc0e8103d588c4634cb74e5851fff"
    "9204cc378948542a8c180f29695de4ab"
)

EXPECTED_SELECTION_COMMIT = (
    "548a175e27326046dee0fd1203f0341147dd4cee"
)


def canonical_sha256(payload):
    return hashlib.sha256(
        json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()


snapshot = json.loads(
    SNAPSHOT_PATH.read_text(
        encoding="utf-8"
    )
)

snapshot_sha = canonical_sha256(
    snapshot
)

assert (
    snapshot_sha
    == EXPECTED_SNAPSHOT_CANONICAL_SHA
)

assert snapshot["record_count"] == 20

assert (
    snapshot["provenance"][
        "selection_manifest_canonical_sha256"
    ]
    == EXPECTED_SELECTION_CANONICAL_SHA
)

assert (
    snapshot["provenance"][
        "selection_manifest_commit"
    ]
    == EXPECTED_SELECTION_COMMIT
)

assert (
    snapshot["provenance"][
        "selection_was_frozen_before_text_access"
    ]
    is True
)

print(
    "Frozen manual-review snapshot verified."
)

print(
    "Snapshot canonical SHA256:",
    snapshot_sha,
)


Frozen manual-review snapshot verified.
Snapshot canonical SHA256: 916ec84712feabdfc0e1feb26c0ba0edd09ad51209aee444e3d7341f6551bff8


In [2]:
annotations = [
    {
        "review_order": 1,
        "source_index": 29330,
        "primary_failure_mode": "possible label-text mismatch / annotation noise",
        "qualitative_interpretation": "The visible review text is strongly positive: it praises the place, cleanliness, relaxation, and overall experience. There is essentially no negative evidence in the supplied text, yet the frozen dataset label is negative. This is therefore better treated as a possible label-text mismatch or missing-context case than as a clear semantic failure of the classifier.",
        "label_alignment_note": "Visible text appears inconsistent with the negative dataset label."
    },
    {
        "review_order": 2,
        "source_index": 14825,
        "primary_failure_mode": "sarcasm plus mixed sentiment",
        "qualitative_interpretation": "The review communicates poor service through extended sarcasm and the Yin/Yang metaphor while also praising the food and ending with superficially positive language. A model relying heavily on local positive lexical patterns can miss that much of the praise is sarcastic and that the overall service judgment is negative.",
        "label_alignment_note": "Dataset label and overall pragmatic meaning appear aligned."
    },
    {
        "review_order": 3,
        "source_index": 21331,
        "primary_failure_mode": "conditional praise and sentiment scope",
        "qualitative_interpretation": "Positive words such as 'awesome' occur repeatedly, but the first occurrence is conditional and the later praise refers to alternative bars rather than the reviewed venue. The actual recommendation is that most people should go elsewhere. The error is consistent with difficulty modeling scope and which entity the positive language describes.",
        "label_alignment_note": "Negative label appears consistent with the review's recommendation."
    },
    {
        "review_order": 4,
        "source_index": 5752,
        "primary_failure_mode": "review-update history and stale positive content",
        "qualitative_interpretation": "The opening note says an earlier four-star review was downgraded because food quality and customer service declined, but most of the remaining body still contains the older highly positive review. The positive historical content greatly outweighs the short update cue, making this a temporal-update and document-composition problem.",
        "label_alignment_note": "The negative label is plausible given the update, although most visible body text reflects the older positive opinion."
    },
    {
        "review_order": 5,
        "source_index": 8426,
        "primary_failure_mode": "mixed aspect sentiment",
        "qualitative_interpretation": "The reviewer is disappointed by several food items and serving-temperature issues, but strongly praises the empanadas, service, drinks, atmosphere, and venue. The review contains substantial evidence for both classes, so an overall negative label requires integrating aspect-level judgments rather than counting positive expressions.",
        "label_alignment_note": "Mixed review; the negative dataset label is plausible but not lexically dominant."
    },
    {
        "review_order": 6,
        "source_index": 22807,
        "primary_failure_mode": "possible temporal-update label/text mismatch",
        "qualitative_interpretation": "The text begins by saying service changed since an earlier post, but almost the entire visible review then describes horrible service and disrespect. The frozen positive label is difficult to reconcile with the supplied body. This looks like a possible rating/update mismatch in which the label may reflect a later revision not fully represented by the remaining text.",
        "label_alignment_note": "Visible text appears substantially more negative than the positive dataset label."
    },
    {
        "review_order": 7,
        "source_index": 30793,
        "primary_failure_mode": "temporal contrast: old negative experience versus current positive judgment",
        "qualitative_interpretation": "The review's final judgment is clearly positive after an ownership change, but it devotes substantial space to a previous terrible experience. Correct classification requires separating past-state complaints from the reviewer's current sentiment.",
        "label_alignment_note": "Positive label matches the review's current overall judgment."
    },
    {
        "review_order": 8,
        "source_index": 30958,
        "primary_failure_mode": "long negative setup followed by positive resolution",
        "qualitative_interpretation": "Most of the review narrates severe problems with the company and contains dense negative language. Near the end, the reviewer explains that an in-store manager resolved the problem, explicitly justifies a five-star rating, and ends with strong praise. The classifier appears vulnerable to long documents where the decisive sentiment reversal occurs late.",
        "label_alignment_note": "Positive label is supported by the explicit final rating rationale."
    },
    {
        "review_order": 9,
        "source_index": 22451,
        "primary_failure_mode": "aspect conflict and rating-context mismatch",
        "qualitative_interpretation": "The food discussion is overwhelmingly negative, but the reviewer explicitly preserves a four-star rating for the Thursday-night experience. The dataset label therefore reflects the overall rating, whereas most sentence-level sentiment in the review concerns disliked food.",
        "label_alignment_note": "Positive label is explained by the explicit four-star contextual rating."
    },
    {
        "review_order": 10,
        "source_index": 25329,
        "primary_failure_mode": "positive endorsement expressed with negative vocabulary",
        "qualitative_interpretation": "The reviewer defends and recommends the credit-building product, but does so using vocabulary about damaged credit, fees, lateness, blame, and financial mistakes. The pragmatic stance is supportive even though many local lexical cues are negative.",
        "label_alignment_note": "Positive label matches the reviewer's overall endorsement."
    },
    {
        "review_order": 11,
        "source_index": 12415,
        "primary_failure_mode": "mixed sentiment near the decision boundary",
        "qualitative_interpretation": "The review begins from a generally positive attitude toward all-you-can-eat sushi but then criticizes this restaurant's rolls, rice, fish quality, and value. Positive category-level language and negative venue-specific language coexist, which is consistent with the model landing exactly at the 0.50 decision boundary.",
        "label_alignment_note": "Negative label appears consistent with the venue-specific conclusion."
    },
    {
        "review_order": 12,
        "source_index": 16620,
        "primary_failure_mode": "ambivalent colloquial sentiment",
        "qualitative_interpretation": "The reviewer alternates between positive observations ('cool', visually attractive, strong drinks) and complaints about crowding, music, and uncertainty about value. The text is intentionally ambivalent and informal, providing weak evidence for either polarity.",
        "label_alignment_note": "Negative label is plausible, but the text is strongly mixed."
    },
    {
        "review_order": 13,
        "source_index": 31278,
        "primary_failure_mode": "short sarcastic complaint",
        "qualitative_interpretation": "The review is very short and expresses dissatisfaction mainly through a rhetorical question and sarcasm rather than through many explicit negative adjectives. Sparse lexical evidence makes a near-threshold prediction understandable.",
        "label_alignment_note": "Negative label matches the pragmatic complaint."
    },
    {
        "review_order": 14,
        "source_index": 37323,
        "primary_failure_mode": "mild mixed review",
        "qualitative_interpretation": "Positive comments about the space and martini are balanced against mediocre food, insufficient seating, and an explicit statement that the reviewer probably would not return regularly. The overall signal is mildly negative rather than strongly polarized.",
        "label_alignment_note": "Negative label is consistent with the weak return intention."
    },
    {
        "review_order": 15,
        "source_index": 18639,
        "primary_failure_mode": "extremely short positive review",
        "qualitative_interpretation": "This review contains only a few sentiment-bearing phrases describing the gardens and views as spectacular. Although the meaning is positive, the very small amount of evidence makes the model probability almost exactly 0.50.",
        "label_alignment_note": "Positive label clearly matches the visible text."
    },
    {
        "review_order": 16,
        "source_index": 12480,
        "primary_failure_mode": "short-review ambiguity / possible label-text mismatch",
        "qualitative_interpretation": "The short review praises the omelette, bacon, and especially the server; the only clear complaint is that the BLT is small for its price. The visible text therefore reads more positive than the frozen negative label, suggesting either a subtle rating decision not fully expressed in text or label-text ambiguity.",
        "label_alignment_note": "Visible text appears more positive than the negative dataset label."
    },
    {
        "review_order": 17,
        "source_index": 20203,
        "primary_failure_mode": "contrast with misleading positive lexical cues",
        "qualitative_interpretation": "The review is negative overall ('not all that', nothing outstanding), but it includes phrases such as best reviews, not bad, good example, great piece of meat, and acceptable service. Correct classification requires resolving contrast and understanding that several positive expressions refer to expectations or alternatives rather than strong praise of the venue.",
        "label_alignment_note": "Negative label matches the overall evaluation."
    },
    {
        "review_order": 18,
        "source_index": 6611,
        "primary_failure_mode": "negative surface language with positive overall stance",
        "qualitative_interpretation": "The review repeatedly mentions diarrhea, greasy food, indigestion, hangovers, and an unpleasant ambience, yet the reviewer explicitly accepts those drawbacks because the restaurant reliably helps and ends by saying they still like it. The final positive stance is easy to lose among many strong negative lexical cues.",
        "label_alignment_note": "Positive label is supported by the repeated willingness to return and final stance."
    },
    {
        "review_order": 19,
        "source_index": 37771,
        "primary_failure_mode": "positive preamble followed by location-specific reversal",
        "qualitative_interpretation": "The review opens with strong general praise for the Cafe Rio brand, then pivots with 'However' to condemn this specific location and explicitly advises readers not to eat there. The error is consistent with difficulty handling contrast, negation, and entity-specific sentiment.",
        "label_alignment_note": "Negative label matches the location-specific conclusion."
    },
    {
        "review_order": 20,
        "source_index": 26587,
        "primary_failure_mode": "sequence truncation plus late sentiment reversal and high OOV",
        "qualitative_interpretation": "This is the clearest structural failure case. The model-visible effective length is exactly the 256-token maximum and the OOV rate is high. The early portion contains extensive praise of the restaurant's history and food, while the explanation for the low rating and deteriorated experience occurs much later in the original review. Because the classifier consumes only the first 256 model-visible tokens, the decisive negative conclusion can be partially or completely outside its input. High OOV content may add further information loss.",
        "label_alignment_note": "Negative label is supported by the later explicit explanation for the poor rating."
    }
]

records_df = pd.DataFrame(
    snapshot["records"]
)

annotations_df = pd.DataFrame(
    annotations
)

assert len(records_df) == 20
assert len(annotations_df) == 20

assert (
    records_df["review_order"].tolist()
    == annotations_df["review_order"].tolist()
)

assert (
    records_df["source_index"].tolist()
    == annotations_df["source_index"].tolist()
)

analysis_df = records_df.merge(
    annotations_df,
    on=[
        "review_order",
        "source_index",
    ],
    how="inner",
    validate="one_to_one",
)

analysis_df["true_sentiment"] = (
    analysis_df["true_label"]
    .map(
        {
            0: "negative",
            1: "positive",
        }
    )
)

analysis_df["predicted_sentiment"] = (
    analysis_df["prediction"]
    .map(
        {
            0: "negative",
            1: "positive",
        }
    )
)

assert len(analysis_df) == 20

category_counts = (
    analysis_df["review_category"]
    .value_counts()
    .to_dict()
)

expected_counts = {
    "confident_false_positive": 5,
    "confident_false_negative": 5,
    "near_threshold_error": 5,
    "slice_specific_error": 5,
}

assert category_counts == expected_counts

print(
    "Manual-analysis table verified."
)

print(
    "Category counts:",
    category_counts,
)


Manual-analysis table verified.
Category counts: {'confident_false_positive': 5, 'confident_false_negative': 5, 'near_threshold_error': 5, 'slice_specific_error': 5}


## Review fields

For every selected example, the notebook retains:

- frozen test/source row identifier;
- true dataset label;
- BiGRU-Attention prediction;
- positive-class probability;
- wrong-class confidence;
- effective model-visible length;
- OOV count and OOV rate;
- frozen robustness-slice memberships;
- exact raw review text;
- a manually authored primary failure-mode hypothesis;
- a qualitative interpretation;
- a note about apparent agreement between the visible text and dataset label.

The qualitative fields are interpretations of the evidence, not additional model outputs.


In [3]:
summary_columns = [
    "review_order",
    "review_category",
    "source_index",
    "true_sentiment",
    "predicted_sentiment",
    "probability",
    "wrong_class_confidence",
    "effective_length",
    "oov_count",
    "oov_rate",
    "primary_failure_mode",
]

display(
    analysis_df[
        summary_columns
    ].sort_values(
        "review_order"
    )
)


,review_order,review_category,source_index,true_sentiment,predicted_sentiment,probability,wrong_class_confidence,effective_length,oov_count,oov_rate,primary_failure_mode
0,1,confident_false_positive,29330,negative,positive,1.000000,1.000000,21,0,0.000000,possible label-text mismatch / annotation noise
1,2,confident_false_positive,14825,negative,positive,0.998535,0.998535,153,2,0.013072,sarcasm plus mixed sentiment
2,3,confident_false_positive,21331,negative,positive,0.998047,0.998047,26,0,0.000000,conditional praise and sentiment scope
3,4,confident_false_positive,5752,negative,positive,0.997559,0.997559,221,1,0.004525,review-update history and stale positive content
4,5,confident_false_positive,8426,negative,positive,0.997559,0.997559,232,1,0.004310,mixed aspect sentiment
5,6,confident_false_negative,22807,positive,negative,0.000229,0.999771,50,0,0.000000,possible temporal-update label/text mismatch
6,7,confident_false_negative,30793,positive,negative,0.000782,0.999218,43,0,0.000000,temporal contrast: old negative experience ver...
7,8,confident_false_negative,30958,positive,negative,0.001561,0.998439,193,0,0.000000,long negative setup followed by positive resol...
8,9,confident_false_negative,22451,positive,negative,0.001782,0.998218,116,0,0.000000,aspect conflict and rating-context mismatch
9,10,confident_false_negative,25329,positive,negative,0.002132,0.997868,92,1,0.010870,positive endorsement expressed with negative v...


# 1. Confident False Positives

These are negative-labeled examples for which BiGRU-Attention predicted the positive class with very high confidence.

The five cases reveal several distinct phenomena rather than one single failure pattern:

1. **Possible label-text disagreement** - one review is visibly positive despite the negative dataset label.
2. **Sarcasm** - negative service criticism is expressed through superficially positive language.
3. **Scope/entity confusion** - positive terms describe conditions or alternative venues rather than the reviewed venue.
4. **Review-history conflict** - a short negative update is combined with a much longer older positive review.
5. **Mixed aspect sentiment** - negative food judgments coexist with substantial praise for service, drinks, and atmosphere.


In [4]:
fp = analysis_df.loc[
    analysis_df["review_category"]
    == "confident_false_positive"
].copy()

assert len(fp) == 5

pd.set_option(
    "display.max_colwidth",
    None,
)

display(
    fp[
        [
            "review_order",
            "source_index",
            "probability",
            "effective_length",
            "raw_text",
            "primary_failure_mode",
            "qualitative_interpretation",
            "label_alignment_note",
        ]
    ]
)


,review_order,source_index,probability,effective_length,raw_text,primary_failure_mode,qualitative_interpretation,label_alignment_note
0,1,29330,1.000000,21,"Wow love the place and everything is very clean and new!\n\nGreat place to come and relax worth a try!\n\nCheers,\n\nEric Van Nguyen\nVisited April 2012",possible label-text mismatch / annotation noise,"The visible review text is strongly positive: it praises the place, cleanliness, relaxation, and overall experience. There is essentially no negative evidence in the supplied text, yet the frozen dataset label is negative. This is therefore better treated as a possible label-text mismatch or missing-context case than as a clear semantic failure of the classifier.",Visible text appears inconsistent with the negative dataset label.
1,2,14825,0.998535,153,"Do you believe in Yin and Yang? The ancient Asian philosophy suggesting polar opposites are interrelated? \nIf you don't, then you best start believing in it if you're headed to Flo's.\n\nThe manifestation of Yin and Yang is clearly present...like 'in your face' present when you're here. You'll see it from the moment you're seated, but you need to be observant...in tune...at one with the universe.\nLook around.\nNotice anything?\n\nLike the couple who was seated five minutes after you were. See how they're now eating something and enjoying their experience?\nAnd what's the opposite manifestation? Your Yin to their Yang?\nYou guessed it. \nYou're boiling mad, no beverage, food or server anywhere on the horizon.\n\nBut don't be mad. Look for the zen in all of this.\nTry to think in terms of the great Yin and Yang. \nCalm yourself by reasoning how you and your date actually sacrificed your good time in order for another couple to be laughing, joking, eating, and now, sipping cocktails as part of their good time.\nThis is the great balance in life.\n\nNow you try.\nGo ahead..it's easy. \n\nAfter dinner, when the bill comes, simply stiff the waiter on the tip.\nYou'll be happy you did, and now, the waiter will be pissed off...just like you were throughout the dining experience.\nYet another example of Yin and Yang.\nBeautiful, isn't it?\nAre you seeing the zen in all of this yet?\n\nOh, and one more thing.\nDespite your poor service, you'll find the food is really quite delightful.\n\nFlo's really does provide the full Asian experience.\nTruly, the manifestation of Yin and Yang.\n\nKudos Flo!",sarcasm plus mixed sentiment,The review communicates poor service through extended sarcasm and the Yin/Yang metaphor while also praising the food and ending with superficially positive language. A model relying heavily on local positive lexical patterns can miss that much of the praise is sarcastic and that the overall service judgment is negative.,Dataset label and overall pragmatic meaning appear aligned.
2,3,21331,0.998047,26,"This place is only awesome if you are one of three things: 1. an Ohio State fan, 2. a metrosexual, or 3. someone trying to hook up with an Ohio State fan or metrosexual...Everyone else, do yourself a favor and go to any one of the many hundreds of awesome bars in old town that will happily accommodate the general public.",conditional praise and sentiment scope,"Positive words such as 'awesome' occur repeatedly, but the first occurrence is conditional and the later praise refers to alternative bars rather than the reviewed venue. The actual recommendation is that most people should go elsewhere. The error is consistent with difficulty modeling scope and which entity the positive language describes.",Negative label appears consistent with the review's recommendation.
3,4,5752,0.997559,221,"NOTE: This was a 4-star review, but the food quality and ESPECIALLY customer service have gone down the tubes. See update below.\nComplaining I can't find a good meatball sub in Phoenix, I was referred to Santisi Brothers. I was told that the meatballs are still made by the brothers' mother, so I was intrigued. \nSantisi Brothers d

# 2. Confident False Negatives

These are positive-labeled examples for which BiGRU-Attention predicted the negative class with very high confidence.

The cases are dominated by **document-level contrast**:

- a possible update/rating mismatch;
- a formerly bad experience followed by a positive current judgment;
- a long complaint narrative followed by a decisive positive resolution;
- severe criticism of one aspect despite an overall positive rating;
- positive endorsement expressed through language about financial problems, fees, blame, and mistakes.

These examples show why document-level sentiment cannot always be inferred from the local density of positive and negative words.


In [5]:
fn = analysis_df.loc[
    analysis_df["review_category"]
    == "confident_false_negative"
].copy()

assert len(fn) == 5

display(
    fn[
        [
            "review_order",
            "source_index",
            "probability",
            "effective_length",
            "raw_text",
            "primary_failure_mode",
            "qualitative_interpretation",
            "label_alignment_note",
        ]
    ]
)


,review_order,source_index,probability,effective_length,raw_text,primary_failure_mode,qualitative_interpretation,label_alignment_note
5,6,22807,0.000229,50,"EDIT: They really did change the service up since I last posted this.\n\nHorrible service.\n\nUsed to be my favorite pizza in the city (at a reasonable price), but I'm rethinking that. We just had an altercation with a server who refused to split a check when we were paying with cash. He then proceeded to disrespect the party at the table, telling us to 'not give him attitude about it.'\n\nSorry Bella Notte, but we're not children. I don't care if you're working hard - it doesn't give you any excuse to disrespect your paying customers like that.",possible temporal-update label/text mismatch,"The text begins by saying service changed since an earlier post, but almost the entire visible review then describes horrible service and disrespect. The frozen positive label is difficult to reconcile with the supplied body. This looks like a possible rating/update mismatch in which the label may reflect a later revision not fully represented by the remaining text.",Visible text appears substantially more negative than the positive dataset label.
6,7,30793,0.000782,43,"This place is so much better since they changed owners.\n\nMy wife and I went when it was the old owners, it was terrible. We waited forever and the food never came before we walked out. People were served before us that walked in after and my wife actually got her soup before me and I sat and waited while they \""""made more\"""".\n\nIt was horrible.\n\nNow its much better. The staff are very friendly, they treat their customers very well and I have nothing but positive things to now say about this place. Its much better with the new owners.",temporal contrast: old negative experience versus current positive judgment,"The review's final judgment is clearly positive after an ownership change, but it devotes substantial space to a previous terrible experience. Correct classification requires separating past-state complaints from the reviewer's current sentiment.",Positive label matches the review's current overall judgment.
7,8,30958,0.001561,193,"Look, we all know Cox sucks. In fact they are a terrible business and their practices are laughable. However they are a necessary evil if you want Internet that isn't garbage that can handle streaming/gaming. The way they handle issues over the phone is worse than Comcast, and for all my east coast brothers/sisters you know what a serious accusation that is. It all started with shitty TV Service. My household went through 4 dvr boxes. 4. That all broke. All of them. 100's of hours of TV. Lost. Every. Single. Time. Their solution? Downgrade from contour. That's their genius solution. So my solution? Upgrade to Dish for TV. PROBLEM SOLVED. However they charged us $200 to disconnect claiming we were on a contract. Nope, disputed that. However to dispute it you have to literally mail a letter to corporate. In the mean time, they cut off my Internet service for pay bills AND THEN DOUBLE CHARGED ME FOR THE INTERNET. WHAT? EXCUSE ME? How does that even make sense? So, an angry phone call and 2 hours on hold, I get the Internet back on because I work from home and to refund the double charged service they have to send a check, via the mail, to a local store. Okay, cool. A week later, let's drive from San tan valley to Gilbert. Once we arrive , turns out the check wasn't mailed. In fact the person helping us via the phone wrote that checks can't be sent to the store, and put a future apology in the notes to us. Thanks Cox cock. So, if you made it this far, why do I give this store 5 stars? The Manager, Mario. That's why. The savior of my people. Stepped in. Saw the crap we went through and listened to my tale. With about 5 minutes of effort , he credited our account for $300. $200 for the bs fee , $50 for the double charge and another $50 just because he was sorry. That's customer servi

# 3. Near-Threshold Errors

The five examples nearest the frozen 0.50 decision threshold are qualitatively different from the confident errors.

They contain:

- genuine mixed sentiment;
- informal and ambivalent language;
- sarcasm expressed with very few explicit polarity words;
- mild criticism balanced by some praise;
- an extremely short positive review with very little evidence.

The near-threshold behavior is therefore consistent with examples where the available model-visible evidence is weak, balanced, or pragmatically indirect.


In [6]:
near = analysis_df.loc[
    analysis_df["review_category"]
    == "near_threshold_error"
].copy()

assert len(near) == 5

display(
    near[
        [
            "review_order",
            "source_index",
            "true_sentiment",
            "predicted_sentiment",
            "probability",
            "threshold_distance",
            "effective_length",
            "raw_text",
            "primary_failure_mode",
            "qualitative_interpretation",
        ]
    ]
)


,review_order,source_index,true_sentiment,predicted_sentiment,probability,threshold_distance,effective_length,raw_text,primary_failure_mode,qualitative_interpretation
10,11,12415,negative,positive,0.500000,0.000000,59,"My friends and I were in the mood for some cheap sushi..and since I was with 5 guy friends, I was not surprised they chose an all-you-can-eat place. As per my profile, I LOVE AYCE sushi. However, Makino doesn't quite satisfy this AYCE sushi lover. The sushi rolls weren't good. They tasted a bit odd. The rice seemed too tough and the quality of the fish wasn't at a top standard. The restaurant is a good size for those dining in groups, but I really would not miss this restaurant. And even though there are other AYCE sushi places that are a bit more pricey, I'd rather pay a bit more to satisfy my tastebuds as well as have no risk to my own health.",mixed sentiment near the decision boundary,"The review begins from a generally positive attitude toward all-you-can-eat sushi but then criticizes this restaurant's rolls, rice, fish quality, and value. Positive category-level language and negative venue-specific language coexist, which is consistent with the model landing exactly at the 0.50 decision boundary."
11,12,16620,negative,positive,0.500000,0.000000,37,blush is a cool place to hang out.. idk if it's a lounge. felt like it.. but with a club crowd? there's not enough places to chill and a million ppl walking around and around each other.. music was very typical vegas which isn't a favorite of mine but i deal.. and overall.. drinks f*'d me up... so maybe paying the price is worth it? idk.. it's real pretty inside though..,ambivalent colloquial sentiment,"The reviewer alternates between positive observations ('cool', visually attractive, strong drinks) and complaints about crowding, music, and uncertainty about value. The text is intentionally ambivalent and informal, providing weak evidence for either polarity."
12,13,31278,negative,positive,0.500000,0.000000,17,How hard is it to get a 2 donut order right?? \n\nOne chocolate glazed and one strawberry glazed donut. That's IT.\n\n\nApparently pretty hard.,short sarcastic complaint,The review is very short and expresses dissatisfaction mainly through a rhetorical question and sarcasm rather than through many explicit negative adjectives. Sparse lexical evidence makes a near-threshold prediction understandable.
13,14,37323,negative,positive,0.500000,0.000000,26,They have a good space but I don't think their using it wisely... The food is okay for a bar but my martini was very good. Adding a bit more seating would be a good idea because when I went there was not enough seating for watching a band. May go back for an event but probably not on a regular basis.,mild mixed review,"Positive comments about the space and martini are balanced against mediocre food, insufficient seating, and an explicit statement that the reviewer probably would not return regularly. The overall signal is mildly negative rather than strongly polarized."
14,15,18639,positive,negative,0.499756,0.000244,7,"You really need a whole day to see this centre. The gardens are spectacular, as are the views.",extremely short positive review,"This review contains only a few sentiment-bearing phrases describing the gardens and views as spectacular. Although the meaning is positive, the very small amount of evidence makes the model probability almost exactly 0.50."


# 4. Slice-Specific Errors

Exactly one disjoint error was frozen for each robustness slice:

1. short review;
2. medium review;
3. long review;
4. contains-negation;
5. high-OOV-rate.

The high-OOV example is especially informative because its effective model-visible length is exactly **256 tokens**, the configured maximum sequence length. Its review begins with extensive positive historical description, while the explanation for the low rating appears much later. This makes **sequence truncation a concrete, evidence-supported limitation of the input representation**: later text cannot influence the classifier once it lies beyond the fixed model-visible sequence.

The other slice examples reinforce the roles of short-context ambiguity, contrast, mixed sentiment, and negation.


In [7]:
slice_specific = analysis_df.loc[
    analysis_df["review_category"]
    == "slice_specific_error"
].copy()

assert len(slice_specific) == 5

slice_columns = [
    "slice_short_reviews",
    "slice_medium_reviews",
    "slice_long_reviews",
    "slice_contains_negation",
    "slice_high_oov_rate",
]

display(
    slice_specific[
        [
            "review_order",
            "source_index",
            "true_sentiment",
            "predicted_sentiment",
            "probability",
            "effective_length",
            "oov_count",
            "oov_rate",
            *slice_columns,
            "raw_text",
            "primary_failure_mode",
            "qualitative_interpretation",
        ]
    ]
)


review_order  source_index true_sentiment predicted_sentiment  \
15            16         12480       negative            positive   
16            17         20203       negative            positive   
17            18          6611       positive            negative   
18            19         37771       negative            positive   
19            20         26587       negative            positive   

    probability  effective_length  oov_count  oov_rate  slice_short_reviews  \
15     0.997070                11          0  0.000000                 True   
16     0.997559                50          0  0.000000                False   
17     0.003111               121          1  0.008264                False   
18     0.997559                31          0  0.000000                False   
19     0.990723               256         13  0.050781                False   

    slice_medium_reviews  slice_long_reviews  slice_contains_negation  \
15                 False               False                    False   
16                  True               False                     True   
17                 False                True                     True   
18                  True               False                     True   
19                 False                True                     True   

    slice_high_oov_rate  \
15                False   
16                False   
17                False   
18                False   
19                 True   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

# Cross-Case Findings

Across the 20 predetermined examples, the most important qualitative patterns are:

### 1. Mixed sentiment and aspect conflict
Several errors combine strong positive and negative statements in the same review. A single document-level label can depend on which aspect the reviewer ultimately weights most heavily.

### 2. Temporal and discourse reversal
Multiple reviews describe a bad past experience before a positive current conclusion, or begin positively before reversing later. Correct classification requires understanding discourse structure rather than treating all sentiment-bearing phrases equally.

### 3. Sarcasm, rhetorical language, and sentiment scope
Some false positives contain positive words whose pragmatic meaning is negative, conditional, sarcastic, or directed toward another entity.

### 4. Rating/text or update ambiguity
A small number of reviews appear difficult to reconcile with their frozen label using the visible text alone. These are documented as possible label-text or review-history mismatches rather than automatically treated as model mistakes.

### 5. Short-input uncertainty
Several near-threshold or slice-specific reviews contain very little model-visible evidence. Short inputs can make the posterior probability unstable around the fixed threshold.

### 6. Fixed-length truncation
The high-OOV slice example has an effective length of exactly 256 tokens while the original review is much longer. The decisive negative discussion occurs late in the document. Since the model cannot consume tokens beyond the configured maximum length, this is a concrete representation limitation rather than merely an abstract hypothesis.

### 7. Negative lexical density can obscure a positive final stance
Several false negatives contain many strongly negative phrases even though the reviewer ultimately recommends the business, preserves a positive rating, or explicitly states a positive final opinion.


# Scientific Boundary

This manual review is **retrospective error characterization**.

It does **not** establish causal explanations of the BiGRU-Attention model's internal reasoning, and it does not constitute a new model-selection experiment.

Most importantly, because these examples come from the already-consumed official test set:

- no model may be retrained because of these observations;
- no hyperparameter may be changed because of these observations;
- the 0.50 threshold must not be changed because of these observations;
- robustness-slice definitions must not be changed because of these observations;
- no new checkpoint may be selected using these observations.

The appropriate use of this analysis is to document limitations, describe representative failure modes, and motivate possible future work on a separate future dataset or validation protocol.
